**Important:** These notebooks are sequential sections of the master notebook. Run them in order in the **same Colab runtime**; restarting the runtime clears required variables. The self-contained master notebook is recommended for a fresh run. The final-test section requires the locked-predictions GitHub commit before unlocking test labels.

## 6. Final training: three fixed seeds and unlabeled test predictions

In [ ]:
with open(ROOT/'results/final_hyperparameters.json') as f: selection=json.load(f)
cfg=selection['config']; seeds=[SEED,SEED+1,SEED+2]
models=[];locked=pd.DataFrame({'original_index':test_idx})
for i,seed in enumerate(seeds):
 model=fit_model(Xdev,ydev,cfg,seed,f'final_seed_{seed}')
 models.append(model)
 import joblib
 joblib.dump(model[0],ROOT/'results'/f'preprocessor_seed_{seed}.joblib')
 torch.save({'state_dict':model[1].state_dict(),'config':cfg,'seed':seed,'parameters':model[2]['parameters']},ROOT/'results'/f'model_seed_{seed}.pt')
 prob=predict_prob(model,Xtest)
 locked[f'prob_seed_{seed}']=prob
 locked[f'prediction_seed_{seed}']=(prob>=.5).astype(int)
locked.to_csv(ROOT/'results/test_predictions_locked.csv',index=False)
print('LOCKED predictions created, without labels. Commit this CSV, config and code to GitHub BEFORE running the next notebook cell.')

# Save learning curves and architecture summary from primary seed
hist=pd.read_csv(ROOT/'results'/f'final_seed_{SEED}_history.csv')
fig,ax=plt.subplots(figsize=(8,4));ax.plot(hist.epoch,hist.train_loss,label='Train BCE');ax.plot(hist.epoch,hist.monitor_loss,label='Early-stop monitor BCE');ax.legend();ax.set_xlabel('Epoch');fig.tight_layout();fig.savefig(ROOT/'figures/final_training_losses.png');plt.show()
fig,ax=plt.subplots(figsize=(8,3));ax.plot(hist.epoch,hist.lr);ax.set(xlabel='Epoch',ylabel='Learning rate');fig.tight_layout();fig.savefig(ROOT/'figures/final_learning_rate.png');plt.show()
with open(ROOT/'results/model_architecture.json','w') as f:json.dump({'hidden':cfg['hidden'],'input_features':models[0][1].net[0].in_features,'learnable_parameters':models[0][2]['parameters']},f,indent=2)


In [ ]:
fig,ax=plt.subplots(figsize=(8,2));ax.axis('off')
items=[f"Input: {models[0][1].net[0].in_features}"]+[f"Hidden: {v}" for v in cfg['hidden']]+['Output: 1 logit']
for i,s in enumerate(items):ax.text((i+.5)/len(items),.5,s,ha='center',bbox={'boxstyle':'round','fc':'#edf2f7'})
ax.set_title(f"MLP architecture — {models[0][2]['parameters']} trainable parameters")
fig.savefig(ROOT/'figures/mlp_architecture.png',bbox_inches='tight');plt.show()


## STOP — Mandatory separate GitHub commit
1. Download/save `results/test_predictions_locked.csv`, `results/final_hyperparameters.json`, configuration and source code to your GitHub repository.
2. Make a **separate commit** of these files **before** looking at test labels or scores.
3. Record the commit hash in your report. Only then change `CONFIRM_LOCKED_COMMIT=False` to `True` in the next cell and execute it.

**Do not run all cells automatically through this checkpoint.** The CSV must have no target label column.


In [ ]:
# DO NOT RUN BEFORE GitHub commit of locked predictions.
# Confirm the commit yourself. This gate prevents accidental early scoring.
CONFIRM_LOCKED_COMMIT=False  # ONLY change after separate GitHub commit
assert CONFIRM_LOCKED_COMMIT, 'First commit test_predictions_locked.csv and code to GitHub; then set CONFIRM_LOCKED_COMMIT=True.'
ytest=y.iloc[test_idx].reset_index(drop=True)  # FIRST test-label access, after confirmation
from sklearn.metrics import ConfusionMatrixDisplay
rows=[]
for i,seed in enumerate(seeds):
 pr=locked[f'prob_seed_{seed}'].to_numpy();rows.append({'model':f'MLP_seed_{seed}',**metrics(ytest,pr)})
 # Primary model graphics
 if i==0:
  fig,ax=plt.subplots();ConfusionMatrixDisplay.from_predictions(ytest,pr>=.5,ax=ax);fig.savefig(ROOT/'figures/mlp_confusion.png');plt.show()
  fpr,tpr,_=roc_curve(ytest,pr);precision,recall,_=precision_recall_curve(ytest,pr)
  fig,ax=plt.subplots();ax.plot(fpr,tpr,label='MLP');ax.set(xlabel='FPR',ylabel='TPR');ax.legend();fig.savefig(ROOT/'figures/mlp_roc.png');plt.show()
  fig,ax=plt.subplots();ax.plot(recall,precision,label='MLP');ax.set(xlabel='Recall',ylabel='Precision');ax.legend();fig.savefig(ROOT/'figures/mlp_pr.png');plt.show()
  boot=[];rng=np.random.default_rng(SEED+900);yy=ytest.to_numpy()
  for _ in range(1000):
   ids=rng.integers(0,len(yy),len(yy))
   if len(np.unique(yy[ids]))<2:continue
   boot.append(metrics(yy[ids],pr[ids]))
  ci={k:np.percentile([b[k] for b in boot],[2.5,97.5]).tolist() for k in ['AP','ROC_AUC','F1']}
  with open(ROOT/'results/bootstrap_ci.json','w') as f:json.dump(ci,f,indent=2)
  print('Bootstrap 95% CI',ci)
  fp=int(((pr>=.5)&(yy==0)).sum());fn=int(((pr<.5)&(yy==1)).sum());print('False positives:',fp,'False negatives:',fn)
# Baselines: training uses development only
for name,est in [('Dummy',DummyClassifier(strategy='prior')),('LogisticRegression',LogisticRegression(max_iter=2000,random_state=SEED))]:
 pp=preprocess(dict(scaler='Standard',clip=False),Xdev);aa=pp.fit_transform(Xdev);bb=pp.transform(Xtest);est.fit(aa,ydev);pr=est.predict_proba(bb)[:,1]
 rows.append({'model':name,**metrics(ytest,pr)})
 if name=='LogisticRegression':
  fig,ax=plt.subplots();ConfusionMatrixDisplay.from_predictions(ytest,pr>=.5,ax=ax);fig.savefig(ROOT/'figures/logistic_confusion.png');plt.show()
  fpr,tpr,_=roc_curve(ytest,pr);fig,ax=plt.subplots();ax.plot(fpr,tpr,label='Logistic Regression');ax.legend();fig.savefig(ROOT/'figures/logistic_roc.png');plt.show()
  prec,rec,_=precision_recall_curve(ytest,pr);fig,ax=plt.subplots();ax.plot(rec,prec,label='Logistic Regression');ax.legend();fig.savefig(ROOT/'figures/logistic_pr.png');plt.show()
result=pd.DataFrame(rows);result.to_csv(ROOT/'results/test_metrics.csv',index=False)
result[result.model.str.startswith('MLP')].to_csv(ROOT/'results/seed_stability_results.csv',index=False)
display(result)

# Descriptive seed stability summary, not independent confidence intervals
seed_rows=result[result.model.str.startswith('MLP')]
seed_rows.select_dtypes(include='number').agg(['mean','std']).to_csv(ROOT/'results/seed_stability_summary.csv')
# Bootstrap table
pd.DataFrame([{'metric':k,'lower_95':v[0],'upper_95':v[1]} for k,v in ci.items()]).to_csv(ROOT/'results/bootstrap_ci.csv',index=False)
# Error analysis (aggregate only)
primary=locked[f'prob_seed_{SEED}'].to_numpy(); pred=(primary>=.5).astype(int)
pd.DataFrame([{'error_type':'false_positive','count':int(((pred==1)&(ytest.to_numpy()==0)).sum())},{'error_type':'false_negative','count':int(((pred==0)&(ytest.to_numpy()==1)).sum())}]).to_csv(ROOT/'results/error_analysis.csv',index=False)
print('Test evaluation completed. Export results/ and figures/ for the Turkish scientific report.')


## 7. Export project files (after scoring)

In [ ]:
import shutil
archive=shutil.make_archive('/content/MLP_Midterm_14928_results','zip',ROOT)
print('Download:',archive)
try:
 from google.colab import files
 files.download(archive)
except ImportError:
 print('Archive saved:',archive)
